# 05 · Revision runs (peer-review controls)

**Settings:** Accelerator **GPU T4 x2**, Internet off, *Save Version → Save & Run All (Commit)*.
**Inputs:** `bbd-code` (the revised version), the output of notebook **01**, and the output of notebook **02**
(its ledgers mark the original runs and tuning as done, so nothing is repeated).

Runs only the `revision` tier, about 3 GPU-hours on two T4s:

| Block | What | Runs | Answers |
| --- | --- | --- | --- |
| R0 | Q at s = 2.35 (CE) and Q-noent at s = 3 (LA), TON_IoT, rho = 100, seed 0 | 2 | models for hardware session 2 (notebook 06) |
| R1 | Q-noent at s = 1, 1.5 and learned; both datasets, all rho, CE and LA, 10 seeds | 360 | is Q equivalent to Q-noent where the Q - C-bounded gap is largest? |
| R2 | classical readout controls C-sin-diag, C-diag, C-sin, C-bounded-init at s = 1, 2, 4; 5 seeds | 720 | what in Q-noent beats s tanh(Vh): the sinusoid, the diagonal readout or the initial gain? |
| R3 | learning-rate sweep (4 values, wd 0) for Q, Q-noent, C-bounded at s = 1, 4; rho = 100; seeds 1-3 | 288 | is the Q - C-bounded gap a tuning artefact? |
| R4 | early stopping on a validation set capped at each class's training count; rho = 100, 1000; 5 seeds | 200 | does the balanced validation set inflate rare-class results? |
| R5 | unbounded head without output bias (C-host-nobias), 10 seeds | 120 | does H1 confound the bound with the output bias? |

Tuning first runs 8 trials for each new head family (240 short classical trials).

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
import json, subprocess, time, torch
from bbd import config, runner
BUDGET_HOURS = float(os.environ.get("BBD_BUDGET_HOURS", 11.3))
TIERS = "revision"
N_WORKERS = max(1, torch.cuda.device_count())
START = time.time(); DEADLINE = START + BUDGET_HOURS * 3600
RESULTS = config.RESULTS_DIR; RESULTS.mkdir(parents=True, exist_ok=True)
n = runner.carry_forward(RESULTS, config.INPUT_ROOT, kinds=("ledger",))   # done-markers only; packs stay in 02's output
print(f"carried forward {n} ledger files; workers: {N_WORKERS}")
from bbd.data import find_prepared_dir
for ds in config.DATASETS:
    print(ds, "prepared at", find_prepared_dir(ds))
print(json.dumps(runner.status(RESULTS, tiers=("revision",)), indent=1))

In [ ]:
def launch(stage, deadline=DEADLINE):
    """Start one worker per GPU, stream progress, return when all workers exit."""
    launch_id = f"{stage}-{int(time.time())}"
    (RESULTS / "logs").mkdir(exist_ok=True)
    procs = []
    for i in range(N_WORKERS):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i) if torch.cuda.is_available() else "",
                   BBD_LAUNCH=launch_id, BBD_THREADS="2")
        log = open(RESULTS / "logs" / f"{launch_id}_w{i}.log", "w")
        cmd = [sys.executable, "-m", "bbd.runner", stage, "--worker", str(i), "--n-workers", str(N_WORKERS),
               "--deadline", str(deadline), "--tiers", TIERS, "--results", str(RESULTS)]
        procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd=CODE), log))
    last = 0
    while any(p.poll() is None for p, _ in procs):
        time.sleep(30)
        if time.time() - last > 600:
            last = time.time()
            print(time.strftime("%H:%M"), json.dumps(runner.status(RESULTS)), flush=True)
            for i in range(N_WORKERS):
                tail = (RESULTS / "logs" / f"{launch_id}_w{i}.log").read_text().splitlines()[-2:]
                print(f"   w{i}:", *tail, sep="\n      ")
    for p, log in procs:
        log.close()
    return [p.returncode for p, _ in procs]

## Tuning of the new head families (skipped once complete)

In [ ]:
st = runner.status(RESULTS, tiers=("revision",))
if st["tuning"][0] < st["tuning"][1]:
    print("tuning exit codes:", launch("tune"))
hps = runner.best_hyperparameters(RESULTS)
import pandas as pd
new = {k: v for k, v in hps.items() if any(m in k for m in ("C-sin", "C-diag", "C-bounded-init", "C-host-nobias"))}
display(pd.DataFrame([dict(key=k, **v) for k, v in sorted(new.items())]))

## Revision grid

In [ ]:
if time.time() < DEADLINE - 600:
    print("grid exit codes:", launch("run"))
print(json.dumps(runner.status(RESULTS, tiers=("revision",)), indent=1))
errs = [r for r in runner.read_records(RESULTS) if r.get("stage") == "error"]
print(len(errs), "errors")
for r in errs[-5:]:
    print(r["run_id"], r["error"]); print(r["traceback"][-600:])

## Quick look (the full analysis is notebook 07)

In [ ]:
from bbd import stats, revision
df = stats.records_frame(RESULTS)
display(revision.pooled_grid(df, "Q", "Q-noent").round(2))
ctl = revision.controls_table(df)
if len(ctl):
    display(ctl[(ctl.b == "C-bounded") & (ctl.s == "all")].round(2))